# AllTrails Export Control Notebook

## Contol

In [1]:
# Control Cell
TEST_TRAIL_URL_EXPORT = False
TEST_TRAIL_GEOMETRY_EXPORT = True
TEST_ACTIVITY_URL_EXPORT = False
TEST_ACTIVITY_EXPORT = False

AUTO_ACTIVITY_URL_EXPORT = False
AUTO_ACTIVITY_EXPORT = False

## Setup

In [2]:
# pip install playwright
# python -m playwright install chromium

import random
import subprocess
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np

# Random pause range (minutes) between automated export calls
MIN_DELAY_MINUTES = 1
MAX_DELAY_MINUTES = 2

In [3]:
REGION_URL = "https://www.alltrails.com/parks/new-zealand/canterbury/banks-peninsula"
TRAIL_URL = "https://www.alltrails.com/trail/new-zealand/canterbury/onawe-pa-track"
ACTIVITY_URL = "https://www.alltrails.com/explore/recording/le-bons-bay-hiking-5f268df"
OUT_DIR = Path("Data\\AllTrails_Downloads")

# Reuse a persistent browser profile so login/challenge can be solved once and reused
# also export storage state after successful access
PROFILE_DIR = Path("Data\\alltrails_profile")
STORAGE_STATE_FILE = Path("Data\\alltrails_storage_state.json")

# URL export writes the shared activity index
INDEX_CSV = OUT_DIR / "activity_index.csv"

# Set this env var in your shell to keep anonymized IDs salted
ID_SALT_ENV = "ALLTRAILS_ID_SALT"

## Testing Calls

In [4]:
if TEST_TRAIL_URL_EXPORT:
    try:
        result = subprocess.run(
            [
                sys.executable,
                "alltrails_trail_urls_export.py",
                "--url",
                REGION_URL,
                "--out-dir",
                str(OUT_DIR),
                "--profile-dir",
                str(PROFILE_DIR),
                "--storage-state",
                str(STORAGE_STATE_FILE),
            ],
            check=True,
            cwd=str(Path.cwd()),
            capture_output=True,
            text=True,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    except subprocess.CalledProcessError as exc:
        if exc.stdout:
            print(exc.stdout)
        if exc.stderr:
            print(exc.stderr)
        raise

In [5]:
if TEST_TRAIL_GEOMETRY_EXPORT:
    # Download GPX files for each trail listed in the Banks Peninsula track URL CSV
    subprocess.run(
        [
            sys.executable,
            "alltrails_trail_data_export.py",
            "--index-csv",
            "Data/AllTrails_Downloads/banks-peninsula_track_urls.csv",
            "--out-dir",
            str(OUT_DIR),
            "--profile-dir",
            str(PROFILE_DIR),
            "--storage-state",
            str(STORAGE_STATE_FILE),
            "--start-row",
            "30",
            "--max-items",
            "35",
            "--retries",
            "1",
            "--menu-timeout-ms",
            "30000",
            "--download-timeout-ms",
            "25000",
            "--slow-mo-ms",
            "5000",
        ],
        check=True,
        cwd=str(Path.cwd()),
    )

In [6]:
if TEST_ACTIVITY_URL_EXPORT:
    try:
        result = subprocess.run(
            [
                sys.executable,
                "alltrails_activity_urls_export.py",
                "--url",
                TRAIL_URL,
                "--out-dir",
                str(OUT_DIR),
                "--profile-dir",
                str(PROFILE_DIR),
                "--storage-state",
                str(STORAGE_STATE_FILE),
                "--index-csv",
                str(INDEX_CSV),
                "--id-salt-env",
                ID_SALT_ENV,
            ],
            check=True,
            cwd=str(Path.cwd()),
            capture_output=True,
            text=True,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    except subprocess.CalledProcessError as exc:
        if exc.stdout:
            print(exc.stdout)
        if exc.stderr:
            print(exc.stderr)
        raise

In [7]:
if TEST_ACTIVITY_EXPORT:
    # Quick sanity run: download 1 activity from the shared index CSV
    subprocess.run(
        [
            sys.executable,
            "alltrails_activity_data_export.py",
            "--index-csv",
            str(INDEX_CSV),
            "--out-dir",
            str(OUT_DIR),
            "--format",
            "csv",
            "--profile-dir",
            str(PROFILE_DIR),
            "--storage-state",
            str(STORAGE_STATE_FILE),
            "--start-row",
            "1",
            "--max-items",
            "1",
            "--retries",
            "1",
            "--menu-timeout-ms",
            "5000",
            "--download-timeout-ms",
            "16000",
            "--slow-mo-ms",
            "0",
        ],
        check=True,
        cwd=str(Path.cwd()),
    )

## Auto Extraction

In [8]:
# banks_peninsula_trail_urls = pd.read_csv("Data/AllTrails_Downloads/banks-peninsula_track_urls.csv")
# banks_peninsula_trail_urls["Extracted"] = pd.DataFrame({'Extracted': [False] * len(banks_peninsula_trail_urls)})
# banks_peninsula_trail_urls.to_csv("Data/AllTrails_Downloads/banks-peninsula_track_urls.csv", index=False)



In [9]:
if AUTO_ACTIVITY_URL_EXPORT:
    banks_peninsula_trail_urls = pd.read_csv("Data/AllTrails_Downloads/banks-peninsula_track_urls.csv")
    for i in range(len(banks_peninsula_trail_urls)):
        
        if banks_peninsula_trail_urls.iloc[i].get("Extracted", 0):
            continue
        else:
            delay_minutes = random.uniform(MIN_DELAY_MINUTES, MAX_DELAY_MINUTES)
            print(f"Waiting {delay_minutes:.2f} minutes before next export...")
            time.sleep(delay_minutes * 60)

            trail_url = banks_peninsula_trail_urls.iloc[i]["track_url"]
            track_name = banks_peninsula_trail_urls.iloc[i]["track_name"]
            print(f"Exporting activity URLs for trail {i+1}/{len(banks_peninsula_trail_urls)}: {track_name}")
            try:
                result = subprocess.run(
                    [
                        sys.executable,
                        "alltrails_activity_urls_export.py",
                        "--url",
                        trail_url,
                        "--out-dir",
                        str(OUT_DIR),
                        "--profile-dir",
                        str(PROFILE_DIR),
                        "--storage-state",
                        str(STORAGE_STATE_FILE),
                        "--index-csv",
                        str(INDEX_CSV),
                        "--id-salt-env",
                        ID_SALT_ENV,
                    ],
                    check=True,
                    cwd=str(Path.cwd()),
                    capture_output=True,
                    text=True,
                )
                if result.stdout:
                    print(result.stdout)
                if result.stderr:
                    print(result.stderr)
                banks_peninsula_trail_urls.loc[i, "Extracted"] = True
            except subprocess.CalledProcessError as exc:
                if exc.stdout:
                    print(exc.stdout)
                if exc.stderr:
                    print(exc.stderr)
                raise
        banks_peninsula_trail_urls.to_csv("Data/AllTrails_Downloads/banks-peninsula_track_urls.csv", index=False)   
            
        
    


In [10]:
if AUTO_ACTIVITY_EXPORT:
    index_df = pd.read_csv(INDEX_CSV)
    start_row = np.where(index_df['download_status'].isna())[0][0]
    
    cmd = [
        sys.executable,
        "-u",
        "alltrails_activity_data_export.py",
        "--index-csv",
        str(INDEX_CSV),
        "--out-dir",
        str(OUT_DIR),
        "--format",
        "csv",
        "--profile-dir",
        str(PROFILE_DIR),
        "--storage-state",
        str(STORAGE_STATE_FILE),
        "--start-row",
        str(start_row),
        "--max-items",
        "5000",
        "--retries",
        "1",
        "--menu-timeout-ms",
        "10000",
        "--download-timeout-ms",
        "16000",
        "--slow-mo-ms",
        "0",
    ]
    
    # Stream child process logs to notebook output as they are produced.
    process = subprocess.Popen(
        cmd,
        cwd=str(Path.cwd()),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)